# Pipeline 1: Naive Single-Pass RAG (Baseline)
### Claim-Verified Multi-Hop RAG — DATASCI 266, Summer 2026

**Purpose:** Establish the baseline. For each HotpotQA question, we:
1. Retrieve the top-k most relevant paragraphs using a **hybrid BM25 + dense retriever** (RRF fusion)
    - RRF --> Reciprocal Rank Fusion (RRF) is a highly effective, parameter-free algorithm used to combine multiple ranked search result lists into a single, unified list
2. Concatenate those paragraphs as context
3. Ask GPT-4o-mini to answer in a single generation step
4. Score with Exact Match and Token-F1

This single-hop approach is the simplest possible RAG system. Notebooks 2 and 3 will build on the retrieval and evaluation infrastructure defined here.

**Dataset:** HotpotQA (`distractor` setting — each question comes with 10 candidate paragraphs: 2 gold + 8 distractors).  
**Retrieval:** Hybrid BM25 + FAISS (sentence-transformers), fused with Reciprocal Rank Fusion (RRF).  
**LLM:** `gpt-4o-mini` via the OpenAI API.  
**Metrics:** Exact Match, Token-F1, Retrieval Recall@k.

In [1]:
# Install all dependencies
# rank_bm25     : fast BM25 implementation
# sentence-transformers : pretrained bi-encoder for dense embeddings
# faiss-cpu     : Facebook's vector similarity search library
# openai        : GPT-4o-mini for generation
# datasets      : HuggingFace datasets (HotpotQA lives here)
# tqdm          : progress bars
%pip install rank_bm25 sentence-transformers faiss-cpu openai datasets tqdm --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 78.2 MB/s eta 0:00:00


In [2]:
import os
import re
import string
import json
import numpy as np
from collections import Counter
from tqdm.auto import tqdm

from datasets import load_dataset
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import faiss
from openai import OpenAI

In [3]:
# OpenAI API key
# In Colab: store your key via Secrets as OPENAI_API_KEY

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

client = OpenAI()

## 1. Load HotpotQA and Build the Corpus

HotpotQA's `distractor` setting is the standard benchmark split. Each example includes:
- `question` — the multi-hop question
- `answer` — the gold answer string
- `context` — a list of 10 `[title, [sentence, ...]]` pairs (2 gold + 8 distractors)
- `supporting_facts` — which sentences are the true evidence

We flatten all paragraphs across the validation set into a single corpus, then index it.
Using the **validation** set (7,405 examples) so we have gold answers to evaluate against.

https://huggingface.co/datasets/hotpotqa/hotpot_qa

In [4]:
# Load HotpotQA — the distractor split is the standard benchmark setting
print("Loading HotpotQA...")
dataset = load_dataset("hotpotqa/hotpot_qa", "distractor")

# Evaluate on the validation set which has gold answers (test set does not)
val_data = dataset["validation"]
print(f"Total validation examples: {len(val_data)}")

# Filter to bridge questions only
# HotpotQA has two question types:
#   - bridge: requires chaining two documents (hop 1 answer → hop 2 retrieval)
#   - comparison: parallel lookup of two facts then compare — no chaining involved
#
# Only bridge questions exhibit the hop-propagation failure mode we're studying
# (a wrong intermediate answer steering later retrieval toward irrelevant evidence).
# Keeping comparison questions would dilute the signal in all three pipelines.
val_data = val_data.filter(lambda x: x["type"] == "bridge")
print(f"Bridge questions only:     {len(val_data)}")
print()

# Inspect one example so we understand the schema
ex = val_data[0]
print("Question:", ex["question"])
print("Answer:  ", ex["answer"])
print("Type:    ", ex["type"])
print("Context paragraphs:", len(ex["context"]["title"]))
print("First title:", ex["context"]["title"][0])
print("First sentences:", ex["context"]["sentences"][0][:2])

Loading HotpotQA...


README.md:   0%|          | 0.00/9.52k [00:00<?, ?B/s]

distractor/train-00000-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/train-00001-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/validation-00000-of-00001.par(…):   0%|          | 0.00/27.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

Total validation examples: 7405


Filter:   0%|          | 0/7405 [00:00<?, ? examples/s]

Bridge questions only:     5918

Question: What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?
Answer:   Chief of Protocol
Type:     bridge
Context paragraphs: 10
First title: Meet Corliss Archer
First sentences: ["Meet Corliss Archer, a program from radio's Golden Age, ran from January 7, 1943 to September 30, 1956.", ' Although it was CBS\'s answer to NBC\'s popular "A Date with Judy", it was also broadcast by NBC in 1948 as a summer replacement for "The Bob Hope Show".']


In [5]:
# Build a flat corpus from all validation paragraphs
# Each entry in the corpus is a single Wikipedia paragraph (title + sentences joined).
# We de-duplicate by (title, text) so that paragraphs shared across questions aren't
# indexed twice, which would inflate dense retrieval scores.

print("Building corpus...")

# Creates an empty list to store the actual text of each unique paragraph.
# corpus_docs[i]  = the full text of paragraph i
corpus_docs   = []

# Creates an empty list to store the title of each unique paragraph at matching indices.
# corpus_titles[i] = its Wikipedia article title
corpus_titles = []

# Creates an empty dictionary to map a unique paragraph (title and text) to its index position in the corpus.
# para_to_idx = dedup map: (title, text) -> corpus index
para_to_idx   = {}

# We store for each val example, the corpus indices of its gold paragraphs.
# This lets us compute Retrieval Recall@k during evaluation.
val_gold_indices = []  # list of sets, one per val example


# Loops through each example in the val set displaying the status bar
# Extracts the list of all available paragraph titles for the current example.
# Extracts the list of sentences (grouped by paragraph) for the current example.
# Converts the list of titles that actually contain the answers into a set for fast lookup.
# Initializes an empty set to store the exact corpus indices of the correct paragraphs for this specific example.
for ex in tqdm(val_data, desc="Indexing paragraphs"):
    titles = ex["context"]["title"]
    sentences = ex["context"]["sentences"]
    gold_titles = set(ex["supporting_facts"]["title"])
    gold_idxs = set()

    # Pairs each paragraph title with its corresponding list of sentences and loops through them.
    # Combines the individual sentences of the current paragraph into a single continuous string.
    # Creates a unique tuple using the title and text to serve as a lookup key.
    for title, sents in zip(titles, sentences):
        text = " ".join(sents)
        key = (title, text)
        # Checks if this specific paragraph has already been added to our global corpus dictionary.
          # Maps the paragraph key to its new index (which equals the current length of the corpus)
          # Adds the new paragraph text to the end of the global text corpus list.
          # Adds the new paragraph title to the end of the global title corpus list.
        if key not in para_to_idx:
            para_to_idx[key] = len(corpus_docs)
            corpus_docs.append(text)
            corpus_titles.append(title)
        # Checks if the title of the current paragraph matches any of the known ground-truth supporting titles.
        # Saves the unique corpus index of this supporting paragraph into the current example's gold set.
        if title in gold_titles:
            gold_idxs.add(para_to_idx[key])
    # Appends the final set of correct paragraph indices for this example to the master validation list.
    val_gold_indices.append(gold_idxs)

# Prints the total number of unique paragraphs found, formatted with commas for readability.
print(f"Corpus size: {len(corpus_docs):,} unique paragraphs")

Building corpus...


Indexing paragraphs:   0%|          | 0/5918 [00:00<?, ?it/s]

Corpus size: 54,391 unique paragraphs


## 2. Build the Hybrid Retriever (BM25 + Dense FAISS)
| Component | What it captures |
|-----------|------------------|
| **BM25** | Exact keyword matches (entity names, rare terms) |
| **Dense (FAISS)** | Semantic similarity (paraphrase, synonyms) |
| **RRF fusion** | Combines both ranked lists without needing to tune score scales |

**BM25:** is a statistical formula that ranks documents based on the exact words they contain relative to the query.

**Dense (FAISS):** Uses an embedding model to convert text into high dimmensional vectors to represtn semantic meaning of the text.

**Reciprocal Rank Fusion (RRF):** Since BM25 returns a ranked list of documents and FAISS returns a seperate ranked list both using different scales, RRF doesn't use raw scores but evaluates on the rank position of each list.

$$ \text{RRF_Score}(d \in D) = \sum_{m \in M} \frac{1}{k + r_m(d)} $$

**Where:**
* $D$ is the set of all documents in the corpus.
* $M$ is the set of retrieval systems (e.g., $\text{BM25}$ and $\text{Dense FAISS}$).
* $r_m(d)$ is the rank position of document $d$ in retriever $m$ (where $r=1$ for the top result).
* $k$ is a constant smoothing parameter (standard baseline value is $60$).

each document gets a score of `1 / (k + rank)` from each retriever, and the scores are summed. `k=60` is the standard default from the original RRF paper.

In [6]:
# BM25 index
# BM25Okapi expects a list of tokenized documents (list of word lists).
print("Building BM25 index...")

# Process every paragraph text (doc) in the corpus, converts to lowercase and splits on whitespace
tokenized_corpus = [doc.lower().split() for doc in corpus_docs]

# Initializes the BM25Okapi scoring model
# Calculates term frequencies and document lengths for the entire tokenized corpus to prepare for fast searching.
bm25 = BM25Okapi(tokenized_corpus)

print("BM25 index ready.")

Building BM25 index...
BM25 index ready.


In [7]:
# Dense FAISS index
# We use 'all-MiniLM-L6-v2': a small, fast bi-encoder that works well for passage retrieval.
# It produces 384-dim embeddings.
# Note: could later experiment with 'all-mpnet-base-v2' for higher quality

# Stores the name of the pre-trained embedding model from Hugging Face.
EMBED_MODEL = "all-MiniLM-L6-v2"

# Sets the number of paragraphs processed at one single time.
# High numbers speed up processing but require more GPU memory.
BATCH_SIZE  = 512   # tune down if you hit OOM

# Downloads, initializes, and loads the selected text-embedding neural network into runtime memory
print(f"Loading sentence-transformer: {EMBED_MODEL}")
embedder = SentenceTransformer(EMBED_MODEL)

# Displays total paragraph count
# Starts the deep learning inference process to convert the raw text strings into arrays of floating-point numbers.
print(f"Encoding {len(corpus_docs):,} paragraphs (this takes a few minutes on GPU)...")
corpus_embeddings = embedder.encode(
    corpus_docs,                # Pass the list of unique text paragraphs to the encoder
    batch_size=BATCH_SIZE,      # Feeds text chunks to the model 512 rows at a time
    show_progress_bar=True,     # Shows progress bar
    convert_to_numpy=True,      # Saves output as NumPy required by FAISS
    normalize_embeddings=True,  # Divides each vector by its length to make it a unit vector. This guarantees that a fast vector multiplication calculation yields exact Cosine Similarity.
)
print(f"Embeddings shape: {corpus_embeddings.shape}")

# Build a flat FAISS index using inner product (cosine, since we normalized)
# Extracts the dimension size (384) from the matrix structure so FAISS knows the vector width.
dim = corpus_embeddings.shape[1]

# Creates an empty, exact-search FAISS index utilizing Inner Product (IP) mathematics.
# Because the data was normalized earlier, this calculates true Cosine Similarity.
index = faiss.IndexFlatIP(dim)

# Places all the generated paragraph vectors into the physical RAM memory of the FAISS engine for active search.
index.add(corpus_embeddings)
print(f"FAISS index built with {index.ntotal:,} vectors.")

Loading sentence-transformer: all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding 54,391 paragraphs (this takes a few minutes on GPU)...


Batches:   0%|          | 0/107 [00:00<?, ?it/s]

Embeddings shape: (54391, 384)
FAISS index built with 54,391 vectors.


In [8]:
# Save corpus + FAISS index to Google Drive
import pickle
from google.colab import drive
drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/datasci266_rag/"
os.makedirs(SAVE_DIR, exist_ok=True)
with open(f"{SAVE_DIR}corpus.pkl", "wb") as f:
    pickle.dump({"docs": corpus_docs, "titles": corpus_titles,
                 "val_gold_indices": val_gold_indices}, f)
faiss.write_index(index, f"{SAVE_DIR}faiss.index")
print(f"Saved to {SAVE_DIR}")

Mounted at /content/drive
Saved to /content/drive/MyDrive/datasci266_rag/


In [9]:
# Hybrid Retrieval via Reciprocal Rank Fusion (RRF)

def bm25_retrieve(query: str, top_n: int = 100) -> list[int]:
    """Return top_n corpus indices ranked by BM25 score."""
    # Preprocess query text exactly like the corpus tokenization step
    tokens = query.lower().split()
    # Calculate BM25 matching scores for all documents in the index
    scores = bm25.get_scores(tokens)
    # Sort indices based on score (argsort yields low-to-high, [::-1] reverses it to high-to-low)
    ranked = np.argsort(scores)[::-1]
    # Slice out the requested number of top candidates and convert to a standard Python list
    return ranked[:top_n].tolist()


def dense_retrieve(query: str, top_n: int = 100) -> list[int]:
    """Return top_n corpus indices ranked by cosine similarity (FAISS)."""
    # Convert query into a normalized 384-dimensional vector embedding matching the corpus format
    q_emb = embedder.encode([query], normalize_embeddings=True, convert_to_numpy=True)
    # Run a k-Nearest Neighbors vector search inside the FAISS index (ignores scores, returns ranks)
    _, indices = index.search(q_emb, top_n)
    # Extract the array from the batch wrapper row and convert to a standard list
    return indices[0].tolist()


def reciprocal_rank_fusion(ranked_lists: list[list[int]], k: int = 60) -> list[int]:
    """
    Combine multiple ranked lists with RRF.

    For each document d appearing at rank r in a list,
    its RRF score = sum over lists of  1 / (k + r).
    k=60 is the standard default (Cormack et al., 2009).
    Returns document indices sorted by descending RRF score.
    """
    # Key = Document Index (int), Value = Accumulative RRF Score (float)
    scores: dict[int, float] = {}
    # Iterate through each system's ranked output list (BM25 list, then FAISS list)
    for ranked in ranked_lists:
        # Loop through document IDs while maintaining the 1-based rank position
        for rank, doc_idx in enumerate(ranked, start=1):
            # Add the reciprocal rank score to the document's total, initializing at 0.0 if new
            scores[doc_idx] = scores.get(doc_idx, 0.0) + 1.0 / (k + rank)
    # Sort the dictionary keys (doc IDs) based on their assigned RRF values in descending order
    return sorted(scores, key=scores.__getitem__, reverse=True)


def hybrid_retrieve(query: str, top_k: int = 5, candidate_n: int = 100) -> list[int]:
    """
    Full hybrid retrieval pipeline.

    1. BM25 retrieves `candidate_n` candidates.
    2. Dense retrieves `candidate_n` candidates.
    3. RRF merges and re-ranks both lists.
    4. Return the top_k corpus indices.

    `candidate_n` controls the recall/speed trade-off for each sub-retriever
    before fusion; 100 is a reasonable default for a ~70k paragraph corpus.
    """
    # 1. Fetch top keyword matches
    bm25_results  = bm25_retrieve(query, top_n=candidate_n)
    # 2. Fetch top vector semantic matches
    dense_results = dense_retrieve(query, top_n=candidate_n)
    # 3. Intersect, score, and re-rank the union of both candidates lists using RRF math
    fused = reciprocal_rank_fusion([bm25_results, dense_results])
    # 4. Return the absolute best final matches requested by downstream application
    return fused[:top_k]


# Quick sanity check
test_q   = val_data[0]["question"]
test_ans = val_data[0]["answer"]
test_res = hybrid_retrieve(test_q, top_k=5)

print(f"Q: {test_q}")
print(f"A: {test_ans}")
print()
print("Top-5 retrieved paragraphs:")
for rank, idx in enumerate(test_res, 1):
    print(f"  [{rank}] {corpus_titles[idx]}: {corpus_docs[idx][:120]}...")

Q: What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?
A: Chief of Protocol

Top-5 retrieved paragraphs:
  [1] A Kiss for Corliss: A Kiss for Corliss is a 1949 American comedy film directed by Richard Wallace and written by Howard Dimsdale.  It stars ...
  [2] Meet Corliss Archer (TV series): Meet Corliss Archer is an American television sitcom that aired on CBS (July 13, 1951 - August 10, 1951) and in syndicat...
  [3] Kiss and Tell (1945 film): Kiss and Tell is a 1945 American comedy film starring then 17-year-old Shirley Temple as Corliss Archer.  In the film, t...
  [4] Evelyn Knight: Evelyn Dawn Knight (born November 5, 1942) is an English woman known for her involvement in the Abscam sting operation o...
  [5] Nora Lewin: Nora Lewin is a fictional character on the TV show "Law & Order", played by two-time Academy Award winning actress Diann...


In [10]:
# Prompt template
# We keep the prompt minimal and consistent across all three pipelines so that
# differences in results reflect the pipeline architecture, not prompt engineering.

# Force strict grounding and a standard fallback string to prevent LLM hallucinations
SYSTEM_PROMPT = """You are a precise question-answering assistant.
Answer the question using ONLY the provided context passages.
Give a short, direct answer (a name, date, number, or brief phrase).
If the context does not contain enough information to answer, respond with exactly: UNANSWERABLE"""


def build_context_string(doc_indices: list[int]) -> str:
    """Format retrieved paragraphs into a numbered context block for the prompt."""
    parts = []
    # Loop over document indices, formatting them into an easy-to-read reference block for the LLM
    for i, idx in enumerate(doc_indices, 1):
        parts.append(f"[{i}] {corpus_titles[idx]}\n{corpus_docs[idx]}")
    # Join documents together with double line breaks for distinct structural spacing
    return "\n\n".join(parts)


def generate_answer(question: str, context: str, model: str = "gpt-4o-mini") -> str:
    """
    Call GPT-4o-mini with the question and retrieved context.

    temperature=0 for deterministic outputs — important for reproducibility
    across evaluation runs.
    """
    user_message = f"""Context passages:
{context}

Question: {question}

Answer:"""
    # Execute the API call to OpenAI's completion endpoint
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_message},
        ],
        temperature=0,    # Eliminates randomness to make sure evaluation results are perfectly replicable
        max_tokens=64,    # Capping length forces conciseness and cuts API token costs
    )
    # Extract the resulting text string and clean off any leading/trailing whitespace noise
    return response.choices[0].message.content.strip()

## 4. Evaluation Metrics

We use the **official HotpotQA evaluation protocol** (same as SQuAD):
- **Exact Match (EM):** 1 if predicted answer == gold after normalization, else 0.
- **Token-F1:** token-level F1 between predicted and gold answer after normalization.

We also track:
- **Retrieval Recall@k:** fraction of examples where at least one gold paragraph is in the top-k results. A low recall means the answer is impossible to derive, setting an upper bound on pipeline accuracy.
- **Abstention rate:** fraction of examples where the model said UNANSWERABLE.

In [11]:
# Normalization (matches official HotpotQA/SQuAD eval script)

def normalize_answer(s: str) -> str:
    """Lowercase, remove punctuation, articles, and extra whitespace."""
    # Uses regex to replace standalone articles ('a', 'an', 'the') with a space
    def remove_articles(text):
        return re.sub(r"\b(a|an|the)\b", " ", text)

    # Strips out duplicate internal spaces, tabs, and newlines
    def white_space_fix(text):
        return " ".join(text.split())

    # Drops all punctuation characters to avoid penalizing missing commas or periods
    def remove_punc(text):
        exclude = set(string.punctuation)
        return "".join(ch for ch in text if ch not in exclude)

    # Standardizes the string by running all cleaning steps in sequence on the lowercase text
    return white_space_fix(remove_articles(remove_punc(s.lower())))

# Returns 1 for an identical cleaned string match, or 0 if they differ
def exact_match(prediction: str, gold: str) -> int:
    """1 if the normalized prediction exactly matches the normalized gold answer."""
    return int(normalize_answer(prediction) == normalize_answer(gold))

# Break down the cleaned prediction and ground-truth strings into lists of words
def token_f1(prediction: str, gold: str) -> float:
    """
    Token-level F1 score.
    Measures partial credit — useful when answers are multi-word phrases
    and the model gets some tokens right.
    """
    pred_tokens = normalize_answer(prediction).split()
    gold_tokens = normalize_answer(gold).split()

    # Finds overlapping tokens by computing the intersection of word frequency counts
    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())

    # Quick exit if there is absolutely no shared vocabulary between the strings
    if num_same == 0:
        return 0.0

    # Calculate Precision (how many predicted words are right) and Recall (how many gold words were caught)
    precision = num_same / len(pred_tokens)
    recall    = num_same / len(gold_tokens)
    # Calculate the harmonic mean of precision and recall
    return 2 * precision * recall / (precision + recall)

# Checks if there is any intersection between the retrieved candidates and the target set
def retrieval_recall_at_k(retrieved_indices: list[int], gold_indices: set[int]) -> int:
    """
    1 if any gold paragraph appears in the retrieved set, else 0.
    'Soft' version — at least one supporting paragraph was found.
    Both gold paragraphs needed for a strict version; we track both.
    """
    return int(len(set(retrieved_indices) & gold_indices) > 0)

# Returns 1 only if every single necessary supporting document is present in your retrieval output
def both_gold_retrieved(retrieved_indices: list[int], gold_indices: set[int]) -> int:
    """1 if ALL gold paragraphs appear in the retrieved set (strict recall)."""
    return int(gold_indices.issubset(set(retrieved_indices)))

## 5. Run the Baseline Pipeline

This section defines and executes the naive single-pass RAG pipeline. The pipeline has three steps for each question:

- **Retrieve** - use the hybrid retriever to fetch the top-k most relevant paragraphs from the corpus
- **Generate** - concatenate the retrieved paragraphs as context and prompt GPT-4o-mini for a direct answer
- **Score** - evaluate the prediction against the gold answer using EM, F1, and retrieval recall

We first define the pipeline functions, then run a smoke test (5.1), an ablation to select the best retrieval configuration (5.2), and finally the full evaluation run that produces the paper results (5.3).

In [12]:
# Retrieval mode switcher
# Single entry point so the pipeline doesn't need to know which retriever is active.
# Pass mode="bm25", "dense", or "hybrid" to swap between them with no other changes.

def retrieve(query: str, top_k: int = 5, mode: str = "hybrid") -> list[int]:
    """
    Unified retrieval interface.

    mode="bm25"   — sparse lexical only (good for entity-name heavy questions)
    mode="dense"  — semantic only (good for paraphrase / synonym heavy questions)
    mode="hybrid" — RRF fusion of both (generally best; professor's recommendation)
    """
    if mode == "bm25":
        return bm25_retrieve(query, top_n=top_k)
    elif mode == "dense":
        return dense_retrieve(query, top_n=top_k)
    elif mode == "hybrid":
        return hybrid_retrieve(query, top_k=top_k)
    else:
        raise ValueError(f"Unknown retrieval mode: {mode!r}. Choose 'bm25', 'dense', or 'hybrid'.")


# Full naive RAG pipeline

def run_naive_rag(
    examples,
    gold_indices_list: list[set[int]],
    top_k: int = 5,
    retrieval_mode: str = "hybrid",
    verbose: bool = False,
) -> list[dict]:
    """
    Run the naive single-pass RAG pipeline on a list of HotpotQA examples.

    Args:
      retrieval_mode: "bm25", "dense", or "hybrid" — controls which retriever is used.
      top_k: number of passages to retrieve and pass as context to the LLM.

    Returns a list of result dicts, one per example, with fields:
      - question, gold_answer, predicted_answer
      - retrieved_indices, em, f1
      - recall_soft (≥1 gold retrieved), recall_strict (all gold retrieved)
      - abstained (model said UNANSWERABLE)
    """
    results = []

    # Iterate through each sample query using a progress bar for time tracking
    for i, ex in enumerate(tqdm(examples, desc=f"naive RAG [{retrieval_mode}, k={top_k}]")):
        question = ex["question"]
        gold = ex["answer"]
        gold_idxs = gold_indices_list[i]

        # Step 1: Retrieve matching documents using the hybrid BM25 + Dense FAISS index
        retrieved = retrieve(question, top_k=top_k, mode=retrieval_mode)

        # Step 2: Build the formatted textual context block and prompt the generation LLM
        context = build_context_string(retrieved)
        predicted = generate_answer(question, context)

        # Step 3: Score accuracy and retrieval effectiveness against standard QA metrics
        em = exact_match(predicted, gold)
        f1 = token_f1(predicted, gold)
        recall_soft = retrieval_recall_at_k(retrieved, gold_idxs)
        recall_strict = both_gold_retrieved(retrieved, gold_idxs)

        # Track whether the LLM correctly identified a lack of information instead of hallucinating
        abstained = int("UNANSWERABLE" in predicted.upper())

        # Consolidate metrics and text info into a structured evaluation logging dictionary
        record = dict(
            question=question,
            gold_answer=gold,
            predicted_answer=predicted,
            retrieved_indices=retrieved,
            em=em,
            f1=f1,
            recall_soft=recall_soft,
            recall_strict=recall_strict,
            abstained=abstained,
        )
        results.append(record)

        # Print detailed individual query summaries dynamically if verbose flag is set
        if verbose:
            status = "✓" if em else "✗"
            print(f"{status} [{i}] Q: {question[:60]}...")
            print(f"    Gold: {gold!r}  |  Pred: {predicted!r}  |  F1: {f1:.2f}")

    return results

# Calculate the average performance metrics across the entire validation subset
def summarize_results(results: list[dict]) -> dict:
    """Aggregate result records into mean metrics."""
    n = len(results)
    metrics = {
        "n": n,
        "EM":                sum(r["em"]           for r in results) / n,
        "F1":                sum(r["f1"]           for r in results) / n,
        "Recall@k (soft)":   sum(r["recall_soft"]  for r in results) / n,
        "Recall@k (strict)": sum(r["recall_strict"] for r in results) / n,
        "Abstention rate":   sum(r["abstained"]    for r in results) / n,
    }
    return metrics

### 5.1 Run the Baseline Smoke Test

We first run a small smoke test (10 examples) to confirm the full pipeline executes end-to-end without errors, verifying the retriever returns results, the prompt formats correctly, the OpenAI API responds, and the scoring functions produce valid output.

Results are printed with verbose=True to see each question, gold answer, and predicted answer individually to check quality.

In [13]:
# Smoke test on 10 examples
# Run this first to confirm the pipeline works end-to-end before spending API budget.

SMOKE_N = 10
smoke_examples = [val_data[i] for i in range(SMOKE_N)]
smoke_gold_indices = val_gold_indices[:SMOKE_N]

smoke_results = run_naive_rag(
    smoke_examples,
    smoke_gold_indices,
    top_k=5,
    verbose=True,
)

smoke_metrics = summarize_results(smoke_results)
print("\n── Smoke test results ──")
for k, v in smoke_metrics.items():
    if k == "n":
        print(f"  {k}: {v}")
    else:
        print(f"  {k}: {v:.3f}")

naive RAG [hybrid, k=5]:   0%|          | 0/10 [00:00<?, ?it/s]

✗ [0] Q: What government position was held by the woman who portrayed...
    Gold: 'Chief of Protocol'  |  Pred: 'UNANSWERABLE'  |  F1: 0.00
✗ [1] Q: What science fantasy young adult series, told in first perso...
    Gold: 'Animorphs'  |  Pred: 'UNANSWERABLE'  |  F1: 0.00
✗ [2] Q: The director of the romantic comedy "Big Stone Gap" is based...
    Gold: 'Greenwich Village, New York City'  |  Pred: 'UNANSWERABLE'  |  F1: 0.00
✓ [3] Q: 2014 S/S is the debut album of a South Korean boy group that...
    Gold: 'YG Entertainment'  |  Pred: 'YG Entertainment'  |  F1: 1.00
✓ [4] Q: Who was known by his stage name Aladin and helped organizati...
    Gold: 'Eenasul Fateh'  |  Pred: 'Eenasul Fateh'  |  F1: 1.00
✗ [5] Q: The arena where the Lewiston Maineiacs played their home gam...
    Gold: '3,677 seated'  |  Pred: 'UNANSWERABLE'  |  F1: 0.00
✗ [6] Q: Who is older, Annie Morton or Terry Richardson?...
    Gold: 'Terry Richardson'  |  Pred: 'UNANSWERABLE'  |  F1: 0.00
✗ [7] Q: What is the name

### 5.2 Ablation Experiment: Retrieval Configuration
Before running the full evaluation, we use a 200-example random subset (seed=42) to select the best retrieval hyperparameters. We sweep across two dimensions:

Retrieval mode: bm25 (sparse lexical), dense (semantic embeddings), or hybrid (RRF fusion of both)
Top-k: 2, 5, or 10 paragraphs passed as context to the LLM

The configuration with the highest Exact Match on this subset is locked in and used for all subsequent runs including the full evaluation in Section 5.3 and both Notebooks 2 and 3.

Holding retrieval fixed across all three pipelines ensures that differences in results reflect the pipeline architecture (single-hop vs. multi-hop vs. claim-verified), not differences in retrieval tuning.

In [14]:
# Sample 200 examples for ablation (fixed across ALL runs)
# seed=42 guarantees the same 200 questions every time this cell runs, so all
# ablation configurations are compared on identical examples.
# Notebooks 2 and 3 will load eval_indices from the saved JSON to ensure
# all three pipelines are also evaluated on the same examples.

EVAL_N = 200
rng = np.random.default_rng(seed=42)
eval_indices = rng.choice(len(val_data), size=EVAL_N, replace=False)

eval_examples = [val_data[int(i)] for i in eval_indices]
eval_gold_indices = [val_gold_indices[int(i)] for i in eval_indices]

print(f"Ablation set: {EVAL_N} examples (seed=42, bridge questions only)")

# Ablation grid
# We sweep retrieval_mode and top_k independently.
# Total API calls: 9 configs × 200 examples = 4,500 calls
# Runtime: ~1 hour.

# To run a single config quickly, comment out the loop and set
# retrieval_mode and top_k directly in run_naive_rag() below.

RETRIEVAL_MODES = ["bm25", "dense", "hybrid"]
TOP_K_VALUES = [2, 5, 10]

ablation_results = {}   # keyed by (mode, top_k)

for mode in RETRIEVAL_MODES:
    for top_k in TOP_K_VALUES:
        key = (mode, top_k)
        print(f"\nRunning: retrieval_mode={mode!r}, top_k={top_k}")
        run_results = run_naive_rag(
            eval_examples,
            eval_gold_indices,
            top_k=top_k,
            retrieval_mode=mode,
        )
        ablation_results[key] = run_results

print("\nAblation complete.")

Ablation set: 200 examples (seed=42, bridge questions only)

Running: retrieval_mode='bm25', top_k=2


naive RAG [bm25, k=2]:   0%|          | 0/200 [00:00<?, ?it/s]


Running: retrieval_mode='bm25', top_k=5


naive RAG [bm25, k=5]:   0%|          | 0/200 [00:00<?, ?it/s]


Running: retrieval_mode='bm25', top_k=10


...


Running: retrieval_mode='hybrid', top_k=5


naive RAG [hybrid, k=5]:   0%|          | 0/200 [00:00<?, ?it/s]


Running: retrieval_mode='hybrid', top_k=10


naive RAG [hybrid, k=10]:   0%|          | 0/200 [00:00<?, ?it/s]


Ablation complete.


In [15]:
# Print ablation table

print(f"{'Mode':<8} {'k':<4} {'EM':>6} {'F1':>6} {'Rec-S':>6} {'Rec-H':>6} {'Abs':>6}")
print("─" * 46)
for (mode, top_k), run_results in sorted(ablation_results.items()):
    m = summarize_results(run_results)
    print(
        f"{mode:<8} {top_k:<4} "
        f"{m['EM']:>6.3f} {m['F1']:>6.3f} "
        f"{m['Recall@k (soft)']:>6.3f} {m['Recall@k (strict)']:>6.3f} "
        f"{m['Abstention rate']:>6.3f}"
    )
print()
print("Rec-S = Recall@k soft (≥1 gold retrieved)")
print("Rec-H = Recall@k strict (both gold retrieved)")


Mode     k        EM     F1  Rec-S  Rec-H    Abs
──────────────────────────────────────────────
bm25     2     0.245  0.310  0.750  0.120  0.550
bm25     5     0.300  0.381  0.850  0.235  0.445
bm25     10    0.315  0.412  0.880  0.330  0.415
dense    2     0.255  0.332  0.830  0.165  0.480
dense    5     0.325  0.420  0.905  0.330  0.420
dense    10    0.320  0.430  0.935  0.405  0.370
hybrid   2     0.275  0.334  0.825  0.130  0.510
hybrid   5     0.330  0.417  0.910  0.315  0.390
hybrid   10    0.365  0.456  0.965  0.450  0.350

Rec-S = Recall@k soft (≥1 gold retrieved)
Rec-H = Recall@k strict (both gold retrieved)


In [20]:
# Pick the best config and lock it in
# This config will be used for the final full-set run and inherited by notebooks 2 and 3.
BEST_MODE  = "hybrid"
BEST_TOP_K = 10

best_results = ablation_results[(BEST_MODE, BEST_TOP_K)]
best_metrics = summarize_results(best_results)

print(f"\nBest config: retrieval_mode={BEST_MODE!r}, top_k={BEST_TOP_K}")
print(f"  EM  : {best_metrics['EM']:.4f}")
print(f"  F1  : {best_metrics['F1']:.4f}")



Best config: retrieval_mode='hybrid', top_k=10
  EM  : 0.3650
  F1  : 0.4562


In [21]:
# Save to Drive + local JSON
# eval_indices saved explicitly so notebooks 2 and 3 can load the exact same
# 200-example subset without regenerating from seed (avoids any ambiguity if
# val_data length ever changes between sessions).

output = {
    "pipeline":       "naive_rag",
    "eval_n":         EVAL_N,
    "seed":           42,
    "eval_indices":   eval_indices.tolist(),   # ← explicit indices for notebooks 2 & 3
    "best_mode":      BEST_MODE,
    "best_top_k":     BEST_TOP_K,
    "embed_model":    EMBED_MODEL,
    "llm":            "gpt-4o-mini",
    "ablation": {
        f"{mode}_k{top_k}": summarize_results(res)
        for (mode, top_k), res in ablation_results.items()
    },
    "best_metrics":   best_metrics,
    "best_results":   best_results,
}

with open("naive_rag_results.json", "w") as f:
    json.dump(output, f, indent=2)

# Also save to Drive so it persists across Colab sessions
with open(f"{SAVE_DIR}naive_rag_results.json", "w") as f:
    json.dump(output, f, indent=2)

print("\nResults saved to naive_rag_results.json and Google Drive.")


Results saved to naive_rag_results.json and Google Drive.


In [22]:
# Error analysis: where does the baseline fail?
# Breaks failures into two categories to understand which problem multi-hop solves:
#   1. Retrieval failure  — gold paragraphs were never retrieved (retrieval ceiling hit)
#   2. Generation failure — gold paragraphs were retrieved but LLM still got it wrong

failures               = [r for r in best_results if r["em"] == 0]
retrieval_ok_but_wrong = [r for r in failures if r["recall_strict"] == 1]
retrieval_failed       = [r for r in failures if r["recall_soft"] == 0]

print(f"Total failures: {len(failures)}")
print(f"Both gold paras retrieved, still wrong: {len(retrieval_ok_but_wrong)}, Generation / reasoning failure")
print(f"Gold paras not retrieved at all:{len(retrieval_failed)}, Retrieval failure (multi-hop needed)")
print()

# Sample retrieval failures — these are the cases that motivate multi-hop RAG
print("Sample retrieval failures (multi-hop motivation):")
for r in retrieval_failed[:3]:
    print(f"Q: {r['question']}")
    print(f"Gold: {r['gold_answer']!r}  |  Pred: {r['predicted_answer']!r}")
    print()

Total failures: 127
Both gold paras retrieved, still wrong: 33, Generation / reasoning failure
Gold paras not retrieved at all:7, Retrieval failure (multi-hop needed)

Sample retrieval failures (multi-hop motivation):
Q: What honor has been received by both a former NASA Astronaut and the Founder of FlightSafety International?
Gold: 'National Aviation Hall of Fame'  |  Pred: 'UNANSWERABLE'

Q: Pluto was the debut album of this artist who also performed on this live music venue that reopened in March 2015.
Gold: 'The Bomb Factory'  |  Pred: 'UNANSWERABLE'

Q: Which number-one single is Michael Steele responsible for?
Gold: '"Walk Like an Egyptian"'  |  Pred: 'UNANSWERABLE'



### 5.3 Full Evaluation Run
With the best retrieval configuration selected from the ablation (Section 5.2), we now run the naive single-pass RAG pipeline on the full bridge-question validation set (~5,918 examples).

Unlike the ablation subset, this run uses every available bridge question to give stable estimates of EM, F1, and retrieval recall. Results are saved to both a local JSON file and Google Drive so they can be loaded directly into the comparison tables in Notebooks 2 and 3 without re-running.

In [23]:
# Full run: best config on the entire bridge validation set

print(f"Running full evaluation: retrieval_mode={BEST_MODE!r}, top_k={BEST_TOP_K}")
print(f"Total examples: {len(val_data):,}")
print("Estimated time: ~3 hours | Estimated cost: ~$1.10\n")

full_results = run_naive_rag(
    list(val_data),
    val_gold_indices,
    top_k=BEST_TOP_K,
    retrieval_mode=BEST_MODE,
    verbose=False,
)

full_metrics = summarize_results(full_results)

print("\n══════════════════════════════════════════")
print("Naive Single-Pass RAG — FINAL RESULTS")
print(f"n = {full_metrics['n']:,} (full bridge validation set)")
print(f"retrieval_mode={BEST_MODE!r}, top_k={BEST_TOP_K}")
print("══════════════════════════════════════════")
for k, v in full_metrics.items():
    if k != "n":
        print(f"  {k:<25}: {v:.4f}")
print("══════════════════════════════════════════")

full_output = {
    "pipeline":   "naive_rag",
    "split":      "full_validation_bridge",
    "n":          len(full_results),
    "best_mode":  BEST_MODE,
    "best_top_k": BEST_TOP_K,
    "embed_model": EMBED_MODEL,
    "llm":        "gpt-4o-mini",
    "metrics":    full_metrics,
    "results":    full_results,
}

with open("naive_rag_full_results.json", "w") as f:
    json.dump(full_output, f, indent=2)

with open(f"{SAVE_DIR}naive_rag_full_results.json", "w") as f:
    json.dump(full_output, f, indent=2)

print("\nFull results saved to naive_rag_full_results.json and Google Drive.")


Running full evaluation: retrieval_mode='hybrid', top_k=10
Total examples: 5,918
Estimated time: ~3 hours | Estimated cost: ~$1.10



naive RAG [hybrid, k=10]:   0%|          | 0/5918 [00:00<?, ?it/s]


══════════════════════════════════════════
Naive Single-Pass RAG — FINAL RESULTS
n = 5,918 (full bridge validation set)
retrieval_mode='hybrid', top_k=10
══════════════════════════════════════════
  EM                       : 0.3403
  F1                       : 0.4459
  Recall@k (soft)          : 0.9542
  Recall@k (strict)        : 0.4633
  Abstention rate          : 0.3552
══════════════════════════════════════════

Full results saved to naive_rag_full_results.json and Google Drive.


## Evalaute if a subset is representative
This does three things:
* Checks whether the 2,200-example subset EM is within 2
points of the full-set EM
* Computes a bootstrap 95% CI
* Saves the exact indices to Drive so pipelines 2 and 3 load the identical examples

In [2]:
import json, numpy as np
from google.colab import drive
drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/datasci266_rag/"
# Load the full pipeline 1 results saved to Drive
with open(f"{SAVE_DIR}naive_rag_full_results.json") as f:
    full_output = json.load(f)

Mounted at /content/drive


In [8]:
# Verify 2,200-example subset is representative of the full set
# Filters the already-saved full pipeline 1 results to a shared eval subset and
# checks whether it's representative before committing API budget to pipelines 2 and 3.
import json, numpy as np

# Load the full pipeline 1 results saved to Drive
with open(f"{SAVE_DIR}naive_rag_full_results.json") as f:
    full_output = json.load(f)
full_results_list = full_output["results"]

# Define the shared eval subset
# This seed and n will be used identically in pipelines 2 and 3.
# Change EVAL_N here if you decide to use a different subset size.
EVAL_N = 2000
EVAL_SEED = 42
rng = np.random.default_rng(seed=EVAL_SEED)
eval_indices_shared = rng.choice(len(full_results_list), size=EVAL_N, replace=False)

# Extract pipeline 1 results for just those examples — no API calls needed
p1_subset = [full_results_list[int(i)] for i in eval_indices_shared]
n = len(p1_subset)

# Compute subset metrics
subset_em            = sum(r["em"]            for r in p1_subset) / n
subset_f1            = sum(r["f1"]            for r in p1_subset) / n
subset_recall_soft   = sum(r["recall_soft"]   for r in p1_subset) / n
subset_recall_strict = sum(r["recall_strict"]  for r in p1_subset) / n
subset_abstention    = sum(r["abstained"]      for r in p1_subset) / n

# Bootstrap 95% CI on EM
# Resamples the subset 1,000 times to estimate the confidence interval on EM.
# This number goes directly into the paper: "EM = X ± Y (95% CI)"
boot_ems = [
    np.mean([p1_subset[i]["em"] for i in np.random.choice(n, n, replace=True)])
    for _ in range(1000)
]
ci_lo, ci_hi = np.percentile(boot_ems, [2.5, 97.5])

# Print comparison and subset metrics
print(f"Pipeline 1 — full set (n={len(full_results_list):,}):       EM={full_output['metrics']['EM']:.4f}")
print(f"Pipeline 1 — subset  (n={n:,}, seed={EVAL_SEED}): EM={subset_em:.4f}  (95% CI: {ci_lo:.3f}–{ci_hi:.3f})")
print(f"Difference: {abs(subset_em - full_output['metrics']['EM']):.4f}")
print()
if abs(subset_em - full_output["metrics"]["EM"]) <= 0.02:
    print("✓ Subset is representative (within 2 points of full set). Safe to use.")
else:
    print("✗ Subset differs by >2 points — try a different seed.")
print()
print("── Pipeline 1 subset metrics (use as P1 column in comparison table) ──")
print(f"  EM                       : {subset_em:.4f}  (95% CI: {ci_lo:.3f}–{ci_hi:.3f})")
print(f"  F1                       : {subset_f1:.4f}")
print(f"  Recall@k (soft)          : {subset_recall_soft:.4f}")
print(f"  Recall@k (strict)        : {subset_recall_strict:.4f}")
print(f"  Abstention rate          : {subset_abstention:.4f}")

# Save shared indices to Drive for pipelines 2 and 3
# Pipelines 2 and 3 load this file instead of regenerating indices, ensuring
# all three pipelines evaluate on the exact same 2,200 examples.
np.save(f"{SAVE_DIR}eval_indices_shared.npy", eval_indices_shared)
print(f"\nShared eval indices saved to {SAVE_DIR}eval_indices_shared.npy")
print("Load in pipelines 2 and 3 with:")
print(f"  eval_indices_shared = np.load(f'{SAVE_DIR}eval_indices_shared.npy')")

Pipeline 1 — full set (n=5,918):       EM=0.3403
Pipeline 1 — subset  (n=2,000, seed=42): EM=0.3385  (95% CI: 0.318–0.361)
Difference: 0.0018

✓ Subset is representative (within 2 points of full set). Safe to use.

── Pipeline 1 subset metrics (use as P1 column in comparison table) ──
  EM                       : 0.3385  (95% CI: 0.318–0.361)
  F1                       : 0.4532
  Recall@k (soft)          : 0.9525
  Recall@k (strict)        : 0.4610
  Abstention rate          : 0.3470

Shared eval indices saved to /content/drive/MyDrive/datasci266_rag/eval_indices_shared.npy
Load in pipelines 2 and 3 with:
  eval_indices_shared = np.load(f'/content/drive/MyDrive/datasci266_rag/eval_indices_shared.npy')


Due to the multi-call structure of the multi-hop pipeline (4 API calls per example) and OpenAI's 10,000 requests-per-day rate limit, evaluating on the full bridge validation set (5,918 examples) would require approximately 3 days of sequential runs. We therefore evaluate all three pipelines on a shared random subset of n=2,000 examples (seed=42), selected by filtering the already-completed Pipeline 1 full-set results and verifying representativeness. The subset EM (0.3385) differs from the full-set EM (0.3403) by only 0.0018, confirming the sample is not a biased draw. At n=2,000, Pipeline 2 requires 8,000 API calls (within the daily limit in a single run with buffer for retries) and Pipeline 3 requires 10,000–12,000 calls depending on whether question decompositions are reused from Pipeline 2. All three pipelines use the identical set of examples loaded from eval_indices_shared.npy, ensuring the comparison is controlled for example selection. This evaluation protocol is consistent with published LLM-in-the-loop multi-hop QA systems, which commonly report results on fixed random subsets when per-example API costs make full-set evaluation impractical.